In [ ]:
# ============================================================================
# CONFIGURAZIONE INIZIALE E DEFINIZIONE DEI SIMBOLI
# ============================================================================
# Questo notebook calcola l'ampiezza per il decadimento B → K π ℓℓ
# utilizzando il formalismo delle ampiezze di elicità con risonanza vettoriale

# --- Import delle librerie necessarie ---
from tqdm.auto import tqdm  # Barra di progresso
import sympy as sp  # Calcolo simbolico
from sympy.physics.quantum.spin import Rotation as rot  # Matrici di rotazione di Wigner
from sympy import pi
from random import randint

# Configurazione per la stampa formattata delle espressioni
sp.init_printing(use_unicode=True, wrap_line=True, num_columns=120)

# --- Parametri di troncamento ---
# Momento angolare massimo
Jmax = 2
Lmax = 2

# --- Definizione dei simboli per i momenti angolari ---
# J: spin della risonanza intermedia
# LB, LR, L1, L2: momenti angolari orbitali
# JX: spin della risonanza fittizia
# lam, lam1, lam2: elicità delle particelle
# eta: chiralità della corrente leptonica (±1)
J, LB, LR, L1, L2, JX, lam, lam1, lam2, eta = sp.symbols('J LB LR L1 L2 JX lam lam1 lam2 eta', integer=True)

# --- Variabili angolari ---
theta1, theta2, thetal, thetaK, phi, chi = sp.symbols('theta1 theta2 thetal thetaK phi chi', real=True)
eta_par = sp.symbols('eta_par', integer=True)  # chiralità per le ampiezze di elicità leptoniche
Res = sp.symbols('Res')  # Etichetta per la risonanza

# --- Funzioni simboliche per le ampiezze di elicità ---
# H: Ampiezza di elicità per la transizione B → K_J (risonanza vettoriale)
H = sp.Function('H', complex=True)(eta, J, lam, LB)
# Hbar: Ampiezza di elicità per la transizione B → K*_J
Hbar = sp.Function('Hbar', complex=True)(eta, J, lam, LB)
# LHA: Ampiezza di elicità leptonica
LHA = sp.Function('LHA', real=True)(eta_par, lam1, lam2)
# G: Ampiezza di elicità effettiva per la transizione K_J → K π
G = sp.Function('G', complex=True)(eta, J, lam, LB)

# --- Propagatore di Breit-Wigner ---
Prop = sp.Function('R')(Res)

# --- Fattori di forma di Blatt-Weisskopf ---
L, rBW = sp.symbols('L rBW', positive=True, real=True)  # L: momento angolare, rBW: raggio
rB, r3, r2 = sp.symbols('rB r3 r2', positive=True, real=True)  # Raggi di interazione
q = sp.symbols('q', real=True)  # Break-up momentum
qR = sp.symbols('qR', real=True)  # Break-up momentum alla risonanza


In [ ]:
# ============================================================================
# COSTRUZIONE DELL'AMPIEZZA TOTALE PER RISONANZA VETTORIALE
# ============================================================================
# Costruisce l'ampiezza totale per il decadimento attraverso una risonanza vettoriale
# B → K*ll → K π ll, dove K* è una risonanza con J=1

# --- Inizializzazione dell'ampiezza ---
AR = 0  # Ampiezza risonante totale
J_val = 1  # Valore dello spin della risonanza (J=1 per risonanza vettoriale)

# --- Definizione delle variabili di elicità ---
lambda1 = sp.symbols('lambda1', real=True)  # Elicità del leptone carico negativo
lambda2 = sp.symbols('lambda2', real=True)  # Elicità del leptone carico positivo

# --- Somma su tutte le elicità possibili della risonanza vettoriale ---
# Per J=1, le elicità possibili sono: -1, 0, +1
for lam_val in range(-min(1,J_val), min(1,J_val) + 1):
    
    # Ampiezza di elicità adronica per la transizione B → K*
    Hterm = H.subs({J: J_val, lam: lam_val, LB: 0})
    
    # --- Termine vettoriale (vector-like) ---
    term_vector = (
        -Hterm * sp.sqrt(3)/(4*sp.pi)  # Fattore di normalizzazione
        * LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2)  # Ampiezza leptonica
        * rot.d(1, lam_val, lambda1 - lambda2, thetal)  # Funzione d di Wigner per i leptoni
        * rot.d(J_val, lam_val, 0, thetaK)  # Funzione d di Wigner per il decadimento K* → Kπ
        * (sp.cos(lam_val*phi) + sp.I*sp.sin(lam_val*phi))  # Fase azimutale e^{i λ φ}
    )
    AR += term_vector

# --- Termine time-like ---
# Contributo dall'operatore di corrente assiale/pseudoscalare
term_timelike = 0
t = sp.symbols('t', real=True)  # polarizzazione time-like
At = sp.symbols('At', complex=True)  # Ampiezza di elicità adronica componente time-like
Lt = sp.Function('Lt', complex=True)(lam1, lam2)  # Ampiezza di elicità leptonica per componente time-like
term_timelike = (
    At * sp.sqrt(3)/(4*sp.pi) *  # Fattore di normalizzazione
    Lt.subs(lam1, lambda1).subs(lam2, lambda2) *
    rot.d(J_val, 0, 0, thetaK)  # Dipendenza angolare solo per λ=0
)

AR += term_timelike

# --- Stampa dell'ampiezza totale ---
print("\n" + "="*50)
if AR == 0:
    print("L'ampiezza AR è nulla (nessuna risonanza selezionata).")
else:
    print("Ampiezza AR totale:")
    print(f"Termini nell'ampiezza AR: {len(AR.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(AR)



Ampiezza AR totale:
Termini nell'ampiezza AR: 4
                  1                                                                    1             1                 ↪
√3⋅At⋅Lt(λ₁, λ₂)⋅d   (thetaK)   √3⋅(-ⅈ⋅sin(φ) + cos(φ))⋅H(η, 1, -1, 0)⋅LHA(η, λ₁, λ₂)⋅d    (thetaK)⋅d          (thetal ↪
                  0,0                                                                  -1,0          -1,λ₁ - λ₂        ↪
───────────────────────────── - ────────────────────────────────────────────────────────────────────────────────────── ↪
             4⋅π                                                          4⋅π                                          ↪

↪                                                          1            1                                              ↪
↪ )   √3⋅(ⅈ⋅sin(φ) + cos(φ))⋅H(η, 1, 1, 0)⋅LHA(η, λ₁, λ₂)⋅d   (thetaK)⋅d         (thetal)   √3⋅H(η, 1, 0, 0)⋅LHA(η, λ₁ ↪
↪                                                          1,0          1,λ₁ - λ₂                      

In [ ]:
# ============================================================================
# SOSTITUZIONE DELLE AMPIEZZE LEPTONICHE E CALCOLO DI |A|²
# ============================================================================
# Sostituiamo le ampiezze di elicità leptoniche con le loro espressioni esplicite
# e calcola il modulo quadro dell'ampiezza totale

# --- Parametri fisici ---
b = sp.symbols('b', real = True)  # beta = sqrt(1 - 4 m_ell^2/q^2)
ml = sp.symbols('ml', real = True)  # Massa dei leptoni

# --- Dizionario delle ampiezze leptoniche esplicite ---
# Basato sulla decomposizione in elicità della corrente leptonica
Leptonic_subs = {
    # --- Ampiezze leptoniche (eta = -1, Left-handed) ---
    # lambda1 = -1/2, lambda2 = +1/2: leptoni con elicità opposte
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (1+b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (1-b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 2*ml,

    # --- Ampiezze leptoniche (eta = +1, Right-handed) ---
    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (1-b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (1+b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 2*ml,

    # --- Ampiezze time-like ---
    Lt.subs({lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    Lt.subs({lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): -2*ml,
    Lt.subs({lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): 0,
    Lt.subs({lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): 0,
}

# --- Calcolo del modulo quadro dell'ampiezza ---
Atot = AR  # Ampiezza totale
Atotsqfin = 0  # Somma su tutte le configurazioni di elicità

# Somma su tutte le possibili elicità dei leptoni
for lambda1_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
    for lambda2_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
        
        # Ampiezza per contributi Left-handed (eta = -1)
        Term_L = Atot.subs({eta: -1, lambda1: lambda1_val, lambda2: lambda2_val})
        # Ampiezza per contributi Right-handed (eta = +1)
        Term_R = Atot.subs({eta: 1, lambda1: lambda1_val, lambda2: lambda2_val})
        
        # Ampiezza totale per questa configurazione di elicità
        Amp_Total_Helicity = Term_L + Term_R
        # Sostituisce le ampiezze leptoniche con le espressioni esplicite
        Amp_Total_Helicity = Amp_Total_Helicity.doit().subs(Leptonic_subs)
        
        # Calcola |A|² = A·A* per questa configurazione
        term_sq = Amp_Total_Helicity * sp.conjugate(Amp_Total_Helicity)
        
        # Accumula il contributo
        Atotsqfin += term_sq

# --- Normalizzazione ---
N = (3*16)/(4*32*sp.pi**2)
Atotsqfin = sp.expand((Atotsqfin/q**2)/N)

# --- Statistiche sull'espressione finale ---
terms_list = Atotsqfin.as_ordered_terms()
print(f"Numero di termini: {len(terms_list)}\n")

# Visualizza l'espressione completa
sp.pprint(Atotsqfin)


Numero di termini: 333

        2    2         __    2                    2       2            2         _______________    2                  ↪
16⋅At⋅ml ⋅cos (thetaK)⋅At   b ⋅H(-1, 1, -1, 0)⋅sin (φ)⋅sin (thetaK)⋅cos (thetal)⋅H(-1, 1, -1, 0)   b ⋅H(-1, 1, -1, 0)⋅ ↪
───────────────────────── + ──────────────────────────────────────────────────────────────────── - ─────────────────── ↪
            2                                                4                                                         ↪
           q                                                                                                           ↪

↪    2       2            2         ______________    2                    2       2            2         ____________ ↪
↪ sin (φ)⋅sin (thetaK)⋅cos (thetal)⋅H(-1, 1, 1, 0)   b ⋅H(-1, 1, -1, 0)⋅sin (φ)⋅sin (thetaK)⋅cos (thetal)⋅H(1, 1, -1,  ↪
↪ ──────────────────────────────────────────────── - ───────────────────────────────────────────────────────────────── ↪
↪      

In [ ]:
# ============================================================================
# TRASFORMAZIONE PER LE FUNZIONI TRIGONOMETRICHE
# ============================================================================
# Converte le potenze di seno e coseno utilizzando identità trigonometriche

# --- Dizionario di trasformazioni trigonometriche ---
ortho_basis = {
    # --- Trasformazioni per θ_K (angolo del kaone) ---
    sp.cos(thetaK)**2: (1 + sp.cos(2*thetaK)) / 2,
    sp.sin(thetaK)**2: (1 - sp.cos(2*thetaK)) / 2,
    sp.sin(thetaK)*sp.cos(thetaK): sp.sin(2*thetaK) / 2,
    
    # --- Trasformazioni per θ_l (angolo leptonico) ---
    sp.cos(thetal)**2: (1 + sp.cos(2*thetal)) / 2,
    sp.sin(thetal)**2: (1 - sp.cos(2*thetal)) / 2,
    sp.sin(thetal)*sp.cos(thetal): sp.sin(2*thetal) / 2,
    
    # --- Trasformazioni per φ (angolo azimutale) ---
    sp.cos(phi)**2: (1 + sp.cos(2*phi)) / 2,
    sp.sin(phi)**2: (1 - sp.cos(2*phi)) / 2,
    sp.sin(phi)*sp.cos(phi): sp.sin(2*phi) / 2,
}

# --- Applica le trasformazioni all'ampiezza ---
Atotsq_mod = Atotsqfin.subs(ortho_basis).expand()


In [ ]:
# ============================================================================
# ESTRAZIONE DELLA STRUTTURA ANGOLARE
# ============================================================================

# --- Funzione di estrazione della struttura trigonometrica ---
def extract_trig_structure(expr):
    """
    Estrae la struttura trigonometrica completa da un'espressione SymPy.
    
    Separa un'espressione in due componenti:
    - trig_factors: fattori che contengono sin e cos
    - non_trig_factors: fattori che non contengono funzioni trigonometriche
    
    Args:
        expr: Espressione SymPy da analizzare
        
    Returns:
        tuple: (trig_factors, non_trig_factors)
    """
    trig_factors = []
    non_trig_factors = []
    
    # Scompone il termine in fattori
    if expr.is_Mul:  # Controlla se è un prodotto 'Mul' per Sympy
        factors = expr.as_ordered_factors()  # Ordina i fattori del prodotto
    elif expr.is_Pow:  # Gestisce le potenze
        base, exp = expr.as_base_exp()  # Divide in base ed esponente
        if base.atoms(sp.sin, sp.cos):  # Se la base contiene funzioni trigonometriche
            return [expr], []  # Tutta la potenza è trigonometrica
        else:
            return [], [expr]  # Tutta la potenza è non-trigonometrica
    else:
        # In questo caso ci sono più termini di tipo 'Add' sommati tra loro
        factors = [expr]  # Li mette in una lista
    
    # Analizza ogni fattore
    for factor in factors:
        if factor.atoms(sp.sin, sp.cos):  # Se contiene sin o cos
            # Questo fattore contiene funzioni trigonometriche
            if factor.is_Pow:
                # È una potenza di funzione trigonometrica (es: sin²θ)
                trig_factors.append(factor)
            elif factor.is_Mul:
                # È un prodotto che contiene trigonometria
                sub_trig, sub_non_trig = extract_trig_structure(factor)
                trig_factors.extend(sub_trig)  # extend aggiunge in coda tutti gli elementi della lista
                non_trig_factors.extend(sub_non_trig)
            else:
                # È una singola funzione trigonometrica (es: sin θ)
                trig_factors.append(factor)
        else:
            # Fattore non trigonometrico (ampiezze adriche, costanti, ecc.)
            non_trig_factors.append(factor)
    
    return trig_factors, non_trig_factors

# --- Raccolta dei coefficienti trigonometrici ---
# Dizionario che mappa prodotti trigonometrici → loro coefficienti
trig_coefficients = {}

print("Analizzando ogni termine dell'espressione...")
# Itera su tutti i termini dell'ampiezza
for i, term in tqdm(enumerate(Atotsq_mod.as_ordered_terms()), 
                     total=len(Atotsq_mod.as_ordered_terms()), 
                     desc="Analisi termini angolari"):
    
    # Estrai la struttura trigonometrica dal termine
    trig_factors, non_trig_factors = extract_trig_structure(term)
    
    # Costruisci il prodotto trigonometrico
    if trig_factors:
        trig_product = 1
        for tf in trig_factors:
            trig_product *= tf
        trig_key = str(trig_product)
    else:
        trig_key = "COSTANTE"
    
    # Costruisci il coefficiente non trigonometrico
    if non_trig_factors:
        non_trig_product = 1
        for ntf in non_trig_factors:
            non_trig_product *= ntf
    else:
        non_trig_product = 1
    
    # Accumula i coefficienti
    if trig_key in trig_coefficients:
        trig_coefficients[trig_key] += non_trig_product
    else:
        trig_coefficients[trig_key] = non_trig_product

print(f"\nTrovati {len(trig_coefficients)} prodotti trigonometrici distinti")

# --- Ordinamento per complessità ---
# Ordina i prodotti trigonometrici per lunghezza della stringa (prima i più semplici)
print("Ordinando i prodotti trigonometrici per complessità")
sorted_keys = sorted(trig_coefficients.keys(), key=lambda x: (len(x), x))

# --- Riepilogo finale ---
print(f"\n" + "="*70)
print("RIEPILOGO FINALE:")
print("="*70)
print(f"- Prodotti trigonometrici distinti: {len([k for k, v in trig_coefficients.items() if v != 0])}")
for k in range(len(sorted_keys)):
    print(f"  • Prodotto #{k}: {sorted_keys[k]}")

print(f"- Analisi completata con successo!")


Analizzando ogni termine dell'espressione...


Analisi termini angolari:   0%|          | 0/458 [00:00<?, ?it/s]


Trovati 18 prodotti trigonometrici distinti
Ordinando i prodotti trigonometrici per complessità

RIEPILOGO FINALE:
- Prodotti trigonometrici distinti: 18
  • Prodotto #0: COSTANTE
  • Prodotto #1: cos(2*phi)
  • Prodotto #2: sin(2*phi)
  • Prodotto #3: cos(thetal)
  • Prodotto #4: cos(2*thetaK)
  • Prodotto #5: cos(2*thetal)
  • Prodotto #6: cos(2*phi)*cos(2*thetaK)
  • Prodotto #7: cos(2*phi)*cos(2*thetal)
  • Prodotto #8: sin(2*phi)*cos(2*thetaK)
  • Prodotto #9: sin(2*phi)*cos(2*thetal)
  • Prodotto #10: cos(2*thetaK)*cos(thetal)
  • Prodotto #11: cos(2*thetaK)*cos(2*thetal)
  • Prodotto #12: sin(2*thetaK)*sin(thetal)*cos(phi)
  • Prodotto #13: sin(phi)*sin(2*thetaK)*sin(thetal)
  • Prodotto #14: sin(2*thetaK)*sin(2*thetal)*cos(phi)
  • Prodotto #15: sin(phi)*sin(2*thetaK)*sin(2*thetal)
  • Prodotto #16: cos(2*phi)*cos(2*thetaK)*cos(2*thetal)
  • Prodotto #17: sin(2*phi)*cos(2*thetaK)*cos(2*thetal)
- Analisi completata con successo!


In [ ]:
# ============================================================================
# CONVERSIONE DA BASE DI ELICITÀ A BASE GENERICA
# ============================================================================

A = sp.Function('A', complex=True)(eta, lam)  # Ampiezza nella base generica
t = sp.symbols('t', real=True)  # Etichetta per componente temporale
perp = sp.symbols('perp', real=True)  # Etichetta per componente perpendicolare
para = sp.symbols('para', real=True)  # Etichetta per componente parallela

# --- Simboli per chiralità Left/Right ---
L = sp.symbols('L', integer=True)  # Chiralità Left-handed
R = sp.symbols('R', integer=True)  # Chiralità Right-handed

# --- Dizionario di sostituzione: H → A ---
# Mappa le ampiezze H(eta, J=1, lam, LB=0) in A(eta, lam)
helicity_to_transverity_subs = {
    # Ampiezze Left-handed (eta = -1)
    H.subs({eta: -1, J: 1, lam: 0, LB: 0}): A.subs({eta: L, lam: 0}),    # Longitudinale
    H.subs({eta: -1, J: 1, lam: 1, LB: 0}): A.subs({eta: L, lam: 1}),    # Elicità +1
    H.subs({eta: -1, J: 1, lam: -1, LB: 0}): A.subs({eta: L, lam: -1}),  # Elicità -1
    
    # Ampiezze Right-handed (eta = +1)
    H.subs({eta: 1, J: 1, lam: 0, LB: 0}): A.subs({eta: R, lam: 0}),     # Longitudinale
    H.subs({eta: 1, J: 1, lam: 1, LB: 0}): A.subs({eta: R, lam: 1}),     # Elicità +1
    H.subs({eta: 1, J: 1, lam: -1, LB: 0}): A.subs({eta: R, lam: -1})    # Elicità -1
}

# --- Applica le sostituzioni a tutti i coefficienti ---
for key in sorted_keys:
    coeff = trig_coefficients[key]
    trig_coefficients[key] = coeff.subs(helicity_to_transverity_subs)


In [ ]:
# ============================================================================
# TRASFORMAZIONE IN BASE DI TRASVERSITÀ
# ============================================================================
# Converte le ampiezze di elicità λ = ±1 nelle combinazioni di trasversità

# --- Dizionario di trasformazione elicità → trasversità ---
transverity_subs = {
    # --- Ampiezze Left-handed ---
    A.subs({eta: L, lam: 1}): (A.subs({eta:L, lam: para}) + A.subs({eta:L, lam: perp}))/sp.sqrt(2),
    A.subs({eta: L, lam: -1}): (A.subs({eta:L, lam: para}) - A.subs({eta:L, lam: perp}))/sp.sqrt(2),
    
    # --- Ampiezze Right-handed ---
    A.subs({eta: R, lam: 1}): (A.subs({eta:R, lam: para}) + A.subs({eta:R, lam: perp}))/sp.sqrt(2),
    A.subs({eta: R, lam: -1}): (A.subs({eta:R, lam: para}) - A.subs({eta:R, lam: perp}))/sp.sqrt(2),    
}

# --- Applica la trasformazione e verifica termini nulli ---
for key in sorted_keys:
    coeff = trig_coefficients[key]
    trig_coefficients[key] = sp.expand(coeff.subs(transverity_subs))


In [ ]:
# ============================================================================
# SOSTITUZIONE DEI PRODOTTI COMPLESSI CON MODULI QUADRATI
# ============================================================================

# Dizionario per memorizzare i coefficienti semplificati
simplified_terms = {}

# --- Costruzione del dizionario di sostituzioni ---
# Mappa i pattern A·conj(A) → |A|² per tutte le combinazioni di eta e lam
conjugate_subs = {}
for x in [0, t, perp, para]:  # Tutte le polarizzazioni trasversità
    for eta_val in [L, R]:  # Left-handed e Right-handed
        conjugate_subs[A.subs({eta: eta_val, lam: x}) * sp.conjugate(A.subs({eta: eta_val, lam: x}))] = sp.Abs(A.subs({eta: eta_val, lam: x}))**2

conjugate_subs[At * sp.conjugate(At)] = sp.Abs(At)**2

# --- Applica le sostituzioni a tutti i coefficienti ---
for key in tqdm(sorted_keys, desc="Semplificazione con dizionario", total=len(sorted_keys)):
    coeff = trig_coefficients[key]
    
    # Salta i coefficienti già nulli
    if coeff == 0:
        simplified_terms[key] = 0
        continue
    
    # Applica le sostituzioni e espande il risultato
    simplified = coeff.subs(conjugate_subs)
    simplified_terms[key] = sp.expand(simplified)


Semplificazione con dizionario:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
# ============================================================================
# SEMPLIFICAZIONE z ± conj(z) → 2Re(z) o 2iIm(z)
# ============================================================================
# Identifica coppie di termini coniugati complessi e li combina nelle loro
# parti reale o immaginaria, semplificando l'espressione finale

# Dizionario per memorizzare i coefficienti finali
FINAL_trig_coefficients = {}

for key in tqdm(sorted_keys, desc="Semplificazione z±conj(z) con dizionario", total=len(sorted_keys)):
    coeff = simplified_terms[key]
    
    if coeff == 0:
        FINAL_trig_coefficients[key] = 0
        continue
    
    # Scompone il coefficiente in termini individuali
    terms = list(coeff.as_ordered_terms())
    
    new_terms = []
    used_indices = set()  # Tiene traccia dei termini già processati
    
    # --- Ricerca di coppie di termini coniugati ---
    for i, term_i in enumerate(terms):
        if i in used_indices:
            continue
        
        found_conjugate = False
        
        # Cerca il coniugato tra gli altri termini
        for j, term_j in enumerate(terms):
            if i == j or j in used_indices:
                continue
            
            # --- Caso 1: z + conj(z) = 2Re(z) ---
            if term_j == sp.conjugate(term_i):
                used_indices.add(i)
                used_indices.add(j)
                found_conjugate = True
                new_terms.append(2 * sp.re(term_i))
                break
            
            # --- Caso 2: z - conj(z) = 2iIm(z) ---
            elif term_j == -sp.conjugate(term_i):
                used_indices.add(i)
                used_indices.add(j)
                found_conjugate = True
                # z - conj(z) è puramente immaginario
                new_terms.append(term_i - sp.conjugate(term_i))
                break
        
        # Se non è stato trovato il coniugato, mantieni il termine originale
        if not found_conjugate:
            new_terms.append(term_i)
            used_indices.add(i)
    
    # --- Ricostruisce il coefficiente finale ---
    if new_terms:
        FINAL_trig_coefficients[key] = sum(new_terms)
    else:
        FINAL_trig_coefficients[key] = 0


Semplificazione z±conj(z) con dizionario:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
# ============================================================================
# RACCOLTA DEI TERMINI SIMILI
# ============================================================================
# Raggruppa i termini simili nei coefficienti, raccogliendo i fattori comuni
# come |A|², Re(A), Im(A) per rendere le espressioni più compatte e leggibili

# Dizionario per i coefficienti con termini raccolti
collected_FINAL_trig_coefficients = {}

# --- Itera su tutti i prodotti trigonometrici ---
for key in tqdm(sorted_keys, desc="Raccolta termini simili", total=len(sorted_keys)):
    coeff = FINAL_trig_coefficients[key]
    
    # Salta i coefficienti già nulli
    if coeff == 0:
        collected_FINAL_trig_coefficients[key] = 0
        continue
    
    # --- Identifica tutti gli atomi che possono essere raccolti ---
    # Estrae tutti i moduli quadri (|A|²)
    abs_terms = list(coeff.atoms(sp.Abs))
    # Estrae tutte le parti reali (Re(...))
    re_terms = list(coeff.atoms(sp.re))
    # Estrae tutte le parti immaginarie (Im(...))
    im_terms = list(coeff.atoms(sp.im))
    
    # Combina tutti i target di raccolta
    collect_targets = abs_terms + re_terms + im_terms
    
    # --- Applica la raccolta ---
    if collect_targets:
        # Raggruppa rispetto a |A|², Re(...), Im(...)
        collected = sp.collect(coeff, collect_targets)
    else:
        # Se non ci sono target da raccogliere, mantieni l'espressione così com'è
        collected = coeff
    
    collected_FINAL_trig_coefficients[key] = collected


Raccolta termini simili:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
# ============================================================================
# DEFINIZIONE DELLE SOSTITUZIONI CINEMATICHE E COSTRUZIONE DI KEY
# ============================================================================

# --- Sostituzioni cinematiche ---
mass = {
    ml**2: (q**2/4)*(1 - b**2)
}
beta = {
    b**2: 1 - 4*ml**2/q**2
}

# --- Costruzione del dizionario KEY ---
# Mappa ogni indice al prodotto trigonometrico × coefficiente corrispondente
KEY = {}
for i, key in enumerate(sorted_keys):
    KEY[i] = sp.simplify(sorted_keys[i]) * collected_FINAL_trig_coefficients[key]


In [ ]:
# ============================================================================
# COMBINAZIONE J1s e J1c
# ============================================================================

J1s1c = KEY[0] + KEY[4]  # Somma dei contributi
index_key = [0, 4]

# Mostra i prodotti trigonometrici coinvolti
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza l'espressione semplificata
sp.pprint(J1s1c.subs(mass).subs(ortho_basis).expand().collect([sp.Abs(A.subs({eta:L, lam:0})), sp.Abs(A.subs({eta:L, lam:para})), sp.Abs(A.subs({eta:L, lam:perp}))]).simplify())


0: COSTANTE
4: cos(2*thetaK)
                                              2   ⎛           __________⎞             2   ⎛           __________⎞      ↪
            2   ⎛        _______⎞   COSTANTE⋅b ⋅re⎝A(L, para)⋅A(R, para)⎠   COSTANTE⋅b ⋅re⎝A(L, perp)⋅A(R, perp)⎠      ↪
- COSTANTE⋅b ⋅re⎝A(L, 0)⋅A(R, 0)⎠ - ───────────────────────────────────── - ───────────────────────────────────── - 2⋅ ↪
                                                      2                                       2                        ↪

↪                               2             2             2             2                                            ↪
↪           2     2   COSTANTE⋅b ⋅│A(R, para)│    COSTANTE⋅b ⋅│A(R, perp)│               ⎛        _______⎞   COSTANTE⋅ ↪
↪ COSTANTE⋅b ⋅│At│  + ───────────────────────── + ───────────────────────── + COSTANTE⋅re⎝A(L, 0)⋅A(R, 0)⎠ + ───────── ↪
↪                                 8                           8                                                        ↪

↪

In [ ]:
# ============================================================================
# COMBINAZIONE J2s e J2c
# ============================================================================

J2s2c = KEY[5] + KEY[11]  # Somma dei contributi
index_key = [5, 11]

# Mostra i prodotti trigonometrici coinvolti
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza l'espressione semplificata
sp.pprint(J2s2c.subs(mass).subs(ortho_basis).expand().collect(sp.Abs(A.subs({eta:L, lam:0}))).simplify())


5: cos(2*thetal)
11: cos(2*thetaK)*cos(2*thetal)
 2 ⎛     2                  2      2                     2      2                     2        2                  2    ↪
b ⋅⎝4⋅sin (thetaK)⋅│A(L, 0)│  + sin (thetaK)⋅│A(L, para)│  + sin (thetaK)⋅│A(L, perp)│  + 4⋅sin (thetaK)⋅│A(R, 0)│  +  ↪
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── ↪
                                                                                                            4          ↪

↪    2                     2      2                     2              2              2⎞              
↪ sin (thetaK)⋅│A(R, para)│  + sin (thetaK)⋅│A(R, perp)│  - 4⋅│A(L, 0)│  - 4⋅│A(R, 0)│ ⎠⋅cos(2⋅thetal)
↪ ────────────────────────────────────────────────────────────────────────────────────────────────────
↪                                                                                                     


In [ ]:
# ============================================================================
# COMBINAZIONE J3
# ============================================================================

J3 = KEY[1] + KEY[6] + KEY[7] + KEY[16]  # Somma dei contributi
index_key = [1, 6, 7, 16]

# Mostra i prodotti trigonometrici coinvolti
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza l'espressione semplificata
sp.pprint(J3.subs(mass).simplify())


1: cos(2*phi)
6: cos(2*phi)*cos(2*thetaK)
7: cos(2*phi)*cos(2*thetal)
16: cos(2*phi)*cos(2*thetaK)*cos(2*thetal)
  2 ⎛            2               2               2               2⎞    2            2                  
-b ⋅⎝│A(L, para)│  - │A(L, perp)│  + │A(R, para)│  - │A(R, perp)│ ⎠⋅sin (thetaK)⋅sin (thetal)⋅cos(2⋅φ) 
───────────────────────────────────────────────────────────────────────────────────────────────────────
                                                   2                                                   


In [ ]:
# ============================================================================
# COMBINAZIONE J4
# ============================================================================

J4 = -KEY[14]  # Singolo contributo (con segno negativo)
index_key = [14]

# Mostra il prodotto trigonometrico coinvolto
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza l'espressione semplificata raccogliendo Re(A_perp)
sp.pprint(J4.subs(mass).expand().collect(sp.re(A.subs({eta:L, lam:perp}))).simplify())


14: sin(2*thetaK)*sin(2*thetal)*cos(phi)
    2 ⎛  ⎛        __________⎞     ⎛        __________⎞⎞                                   
√2⋅b ⋅⎝re⎝A(L, 0)⋅A(L, para)⎠ + re⎝A(R, 0)⋅A(R, para)⎠⎠⋅sin(2⋅thetaK)⋅sin(2⋅thetal)⋅cos(φ)
──────────────────────────────────────────────────────────────────────────────────────────
                                            2                                             


In [ ]:
# ============================================================================
# COMBINAZIONE J5
# ============================================================================

J5 = KEY[12]  # Singolo contributo
index_key = [12]

# Mostra il prodotto trigonometrico coinvolto
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza l'espressione semplificata usando la sostituzione beta
sp.pprint(J5.subs(beta).expand().collect(sp.re(A.subs({eta:L, lam:perp}))).simplify())


12: sin(2*thetaK)*sin(thetal)*cos(phi)
     ⎛  ⎛        __________⎞     ⎛        __________⎞⎞                                 
√2⋅b⋅⎝re⎝A(L, 0)⋅A(L, perp)⎠ - re⎝A(R, 0)⋅A(R, perp)⎠⎠⋅sin(2⋅thetaK)⋅sin(thetal)⋅cos(φ)


In [ ]:
# ============================================================================
# COMBINAZIONE J6
# ============================================================================

J6 = -(KEY[3] + KEY[10])  # Somma con segno negativo
index_key = [3, 10]

# Mostra i prodotti trigonometrici coinvolti
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza l'espressione semplificata raccogliendo Re(A_0)
sp.pprint(J6.subs(mass).expand().collect(sp.re(A.subs({eta:L, lam:0}))).simplify())


3: cos(thetal)
10: cos(2*thetaK)*cos(thetal)
    ⎛  ⎛           __________⎞     ⎛           __________⎞⎞    2                    
2⋅b⋅⎝re⎝A(L, para)⋅A(L, perp)⎠ - re⎝A(R, para)⋅A(R, perp)⎠⎠⋅sin (thetaK)⋅cos(thetal)


In [ ]:
# ============================================================================
# COMBINAZIONE J7
# ============================================================================

J7 = KEY[13]  
index_key = [13]

# Mostra il prodotto trigonometrico coinvolto
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza raccogliendo le parti immaginarie delle interferenze A_perp * A_para*
sp.pprint(J7.subs(mass).expand().collect([sp.im(A.subs({eta:L, lam:perp})*sp.conjugate(A.subs({eta:L, lam:para}))), sp.im(A.subs({eta:R, lam:perp})*sp.conjugate(A.subs({eta:R, lam:para})))]).simplify())


13: sin(phi)*sin(2*thetaK)*sin(thetal)
     ⎛  ⎛        __________⎞     ⎛        __________⎞⎞                                 
√2⋅b⋅⎝im⎝A(L, 0)⋅A(L, para)⎠ - im⎝A(R, 0)⋅A(R, para)⎠⎠⋅sin(φ)⋅sin(2⋅thetaK)⋅sin(thetal)


In [ ]:
# ============================================================================
# COMBINAZIONE J8
# ============================================================================

J8 = -KEY[15] 
index_key = [15]

# Mostra il prodotto trigonometrico coinvolto
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza l'espressione semplificata raccogliendo m_l²
sp.pprint(J8.subs(mass).expand().collect(ml**2).simplify())


15: sin(phi)*sin(2*thetaK)*sin(2*thetal)
    2 ⎛  ⎛        __________⎞     ⎛        __________⎞⎞                                   
√2⋅b ⋅⎝im⎝A(L, 0)⋅A(L, perp)⎠ + im⎝A(R, 0)⋅A(R, perp)⎠⎠⋅sin(φ)⋅sin(2⋅thetaK)⋅sin(2⋅thetal)
──────────────────────────────────────────────────────────────────────────────────────────
                                            2                                             


In [ ]:
# ============================================================================
# COMBINAZIONE J9
# ============================================================================

J9 = -(KEY[2] + KEY[8] + KEY[9] + KEY[17])  # Somma con segno negativo
index_key = [2, 8, 9, 17]

# Mostra i prodotti trigonometrici coinvolti
for i in index_key:
    print(f"{i}: {sorted_keys[i]}")

# Visualizza l'espressione semplificata
sp.pprint(J9.subs(mass).expand().simplify())


2: sin(2*phi)
8: sin(2*phi)*cos(2*thetaK)
9: sin(2*phi)*cos(2*thetal)
17: sin(2*phi)*cos(2*thetaK)*cos(2*thetal)
 2 ⎛  ⎛           __________⎞     ⎛           __________⎞⎞             2            2        
b ⋅⎝im⎝A(L, para)⋅A(L, perp)⎠ + im⎝A(R, para)⋅A(R, perp)⎠⎠⋅sin(2⋅φ)⋅sin (thetaK)⋅sin (thetal)
